# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load Data
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# Prepare Features
def prepare_features(data):
    d = data.copy()
    d["has_word_count"] = d["word_count"].notna().astype(int)
    d["word_count"] = d["word_count"].fillna(0)
    d["has_position_data"] = (d["avg_position"] > 0).astype(int)
    d["log_impressions_90d"] = np.log1p(d["impressions_90d"])
    d["log_clicks_90d"] = np.log1p(d["clicks_90d"])
    return d

df_prep = prepare_features(df)
features = [
    "days_since_last_update", 
    "log_impressions_90d",
    "log_clicks_90d",
    "avg_position", 
    "has_position_data",
    "ctr",
    "word_count",
    "has_word_count",
    "content_age_days"
]
target = "is_declining"

# Time-aware split to get our actionable queue (we act on the most recent content)
df_time = df_prep.sort_values("content_age_days", ascending=False).reset_index(drop=True)
split_idx = int(len(df_time) * 0.8)
df_train = df_time.iloc[:split_idx]
df_test = df_time.iloc[split_idx:].copy()

X_train = df_train[features]
y_train = df_train[target]
X_test = df_test[features]

scaler = StandardScaler()
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(scaler.fit_transform(X_train), y_train)

probs = lr.predict_proba(scaler.transform(X_test))[:, 1]
df_test["decline_probability"] = probs

# Create reason codes and archetypes
def assign_action(row):
    if row['decline_probability'] > 0.75 and row['impressions_90d'] > 1000:
        return 'High Priority Refresh', f"High decline risk ({row['decline_probability']:.1%}); strong historical impressions ({row['impressions_90d']:.0f})"
    elif row['decline_probability'] > 0.75:
        return 'Standard Refresh Review', f"High decline risk ({row['decline_probability']:.1%}); lower volume"
    elif row['impressions_90d'] > 5000 and row['days_since_last_update'] > 180:
        return 'Preventative Update', f"Stale ({row['days_since_last_update']:.0f} days) but high traffic; refresh before decay"
    else:
        return 'Monitor', 'Stable or low impact'

df_test[['recommended_action', 'reason_code']] = df_test.apply(assign_action, axis=1, result_type='expand')

# Filter out "Monitor" for the final queue
action_queue = df_test[df_test['recommended_action'] != 'Monitor'].sort_values('decline_probability', ascending=False)
action_queue_display = action_queue[['content_id', 'decline_probability', 'recommended_action', 'reason_code']].head(10)
display(action_queue_display)

,content_id,decline_probability,recommended_action,reason_code
26572,content_c82bc0c24241,0.931792,High Priority Refresh,High decline risk (93.2%); strong historical i...
27442,content_d6e1bbb4a996,0.910298,High Priority Refresh,High decline risk (91.0%); strong historical i...
24987,content_134631e65b9e,0.909242,High Priority Refresh,High decline risk (90.9%); strong historical i...
28747,content_823ea9b9b355,0.908772,High Priority Refresh,High decline risk (90.9%); strong historical i...
26694,content_c65ee459f729,0.908722,High Priority Refresh,High decline risk (90.9%); strong historical i...
24313,content_3e79eaafc89d,0.907849,High Priority Refresh,High decline risk (90.8%); strong historical i...
28754,content_c90bfc85694f,0.903563,High Priority Refresh,High decline risk (90.4%); strong historical i...
24356,content_477f7892c1f1,0.898813,High Priority Refresh,High decline risk (89.9%); strong historical i...
25653,content_17df7f1f7868,0.896863,High Priority Refresh,High decline risk (89.7%); strong historical i...
25820,content_ef731e95e774,0.896799,High Priority Refresh,High decline risk (89.7%); strong historical i...


### Archetype → Action Mapping & Cost/Value Thinking

**Rule-Based Categories (not semantic clustering):**
1. **High Priority Refresh:** Pages with strong historical visibility (>1000 impressions) showing a high model probability of decline. Action: Comprehensive editorial review and update.
2. **Preventative Update:** Pages that are stale (>180 days) but currently retain high traffic. Action: Light refresh to update facts, dates, and links before decay hits.
3. **Standard Refresh Review:** Pages with high decline risk but lower historical volume. Action: Evaluate if the topic warrants the effort; otherwise, consolidate or leave alone.

**Cost/Value Prioritization:**
- **High Potential / Low Effort:** Preventative updates on evergreen pages (they just need light fact-checking).
- **High Potential / High Effort:** High Priority Refresh pages that need substantial rewriting to match shifting search intent.
- **Low Potential / Low Effort:** Minor formatting tweaks on low-traffic pages.
- **Low Potential / High Effort:** Rewriting pages with zero historical demand and high competition. Skip these.

### Decay / Refresh Insight
In this data, we observed that content age and staleness are strongly associated with a higher likelihood of decline. However, the model also ranks pages with very high historical impressions as likely to drop, suggesting a regression to the mean or the model over-indexing on past visibility. These pages look worth reviewing first because protecting existing visibility is often more reliable than building new visibility from scratch.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who should use this:** Content editors, SEO strategists, and editorial managers.
**What decision it supports:** Prioritizing the content refresh calendar by identifying pages that show a directional risk of traffic decay.
**Intended Use:** This queue provides decision-support. It surfaces candidates that warrant a human look; it is not an automatic publishing system.

**Limits of the Model & Dataset:**
- **Observational Data:** This dataset is observational. We cannot claim that refreshing a page guarantees a traffic improvement or causes a ranking increase.
- **Feature Limitations:** The model relies on historical numeric metrics (like impressions and staleness) and cannot read actual text quality or measure nuance in search intent.
- **Human Review is Necessary:** The algorithm flags numerical decay patterns, not editorial truth. A human must verify the actual context of the page.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human Review Rules:**
Before taking action on any flagged page, an editor MUST check:
- Whether the page actually needs updating (are the facts or examples actually outdated?).
- Search intent: Does the page still answer what a user searching for this topic wants to know?
- Factual accuracy and content quality.
- Business relevance: Does this topic still align with current business goals?
- Whether the recommendation makes sense in context (e.g., if a page was published yesterday, a high risk score might just be a data anomaly).

**The No-Go List (What should NEVER be automated):**
- Automatically publishing content changes generated by AI.
- Automatically deleting or unpublishing pages based on a low score or high decline probability.
- Automatically changing factual claims, numbers, or dates.
- Automatically changing canonical or indexing decisions (e.g., applying noindex tags).
- Automatically applying large-scale SEO structural changes based solely on this score.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Monitoring Triggers (Proposed):**
- **Data Distribution Changes:** If the distribution of impressions_90d or avg_position shifts significantly across the portfolio (e.g., a major algorithm update happens).
- **Feature Missingness Increases:** If Google Search Console disconnects and avg_position or impressions become completely missing for a large batch of active pages.
- **Score Distribution Shifts:** If the model starts predicting >90% of the portfolio as "declining", indicating a systemic drift.
- **Performance Drops:** If the precision of the model on recent holdout data drops below the Week 4 baseline (e.g., Precision@50 falls below 0.40).
- **New Content Types:** If a new client is onboarded with a fundamentally different content strategy (e.g., entirely user-generated content instead of editorial articles).

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [2]:
import os

# Create outputs directory if it doesn't exist
os.makedirs('../outputs', exist_ok=True)

# Export the actionable queue
export_cols = ['content_id', 'decline_probability', 'recommended_action', 'reason_code']
action_queue[export_cols].to_csv('../outputs/content_action_queue.csv', index=False)

print("Successfully exported queue to work/outputs/content_action_queue.csv")

Successfully exported queue to work/outputs/content_action_queue.csv
